# DRAGNET — designed-coalition validation + extraction arms (Kaggle)

Builds the chain-split (AND) and hard-traps (OR) cells, validates the designed coalitions
against model behavior, and compares the five extraction arms on query budget vs coverage.

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100) · Internet **on**.
Both repos clone straight from GitHub; re-running the setup cell pulls the latest.

Outputs land in `runs/` and are zipped for download at the end. Rough budget on a T4:
pipeline ~40–60 min per condition, validation ~1 h, extraction ~2–3 h at the default limits —
trim the knobs in cell 2 if the session gets tight.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
os.chdir(WORK)

def fetch(name, url):
    if (WORK / name).exists():
        subprocess.run(['git', '-C', name, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, name], check=True)

fetch('lineup', 'https://github.com/santoshcheethiralame-dot/LINEUP')

# scope: an attached repo dataset (if any) takes precedence over the clone, for offline runs.
dragnet_src = next((p.parent for p in Path('/kaggle/input').glob('*/**/src/dragnet/__init__.py')), None)
if dragnet_src is not None:
    shutil.rmtree(WORK / 'dragnet', ignore_errors=True)
    shutil.copytree(dragnet_src.parent.parent, WORK / 'dragnet')
else:
    fetch('dragnet', 'https://github.com/santoshcheethiralame-dot/DRAGNET')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './lineup[gpu]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', './dragnet'], check=True)
# The container's transformers refuses 4-bit below this floor; pin it regardless of what resolved.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'], check=True)

# cwd is /kaggle/working, which holds the cloned lineup/ and scope/ folders. In the notebook
# kernel a bare `import lineup` binds to that folder (a namespace package with no submodules) and
# shadows the pip-installed package -> `lineup.data` then fails. Put the real package dirs first.
for pkg in ('lineup', 'dragnet'):
    src = str(WORK / pkg / 'src')
    if src not in sys.path:
        sys.path.insert(0, src)

import bitsandbytes
print('[ok] lineup + dragnet installed; bitsandbytes', bitsandbytes.__version__)

In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
TAG = 'qwen'
DATASET = 'hotpotqa'          # hotpotqa | 2wiki | musique
LIMIT = 300                   # source questions per condition (chain-split skips non-bridge cases)
K = 6
SEED = 0
SUFFIX = ''                   # e.g. '_k10' on a retrieval-depth run so sweep zips do not collide
RUN_AND = True                # chain-split condition (H3/H4; the AND cells)
RUN_OR = False                # hard-traps OR cover; NOT a prereg hypothesis and the heaviest stage
                              # (~3-4 h of LOO oracle) -> off by default so a run fits 12 h with margin.
                              # Run it as its own commit (RUN_AND/RUN_NATURAL=False) when you want it.
RUN_NATURAL = True            # organic errors, nothing planted: the MSCS target (H1/H2/H5)
RUN_ORDERS = True             # on the natural cell, log DRAGNET's interaction/shapley orders -> the real H2
REAL_RETRIEVER = False        # natural cell draws distractors from a live BM25 retriever (use TAG like 'qwenbm25')
VALIDATE_MAX_SIZE = 3         # exact-enumeration bound for validate_designed
VALIDATE_LIMIT = 60           # wrong cases per cell for validation (0 = all)
EXTRACT_LIMIT = 40            # wrong cases per cell for the extraction arms (0 = all)
N_SAMPLES = 48                # surrogate masks for the interaction/beam/orders arms
MSCS_MAX_SIZE = 3             # enumerated set-size bound on the natural cell
MSCS_LIMIT = 100              # wrong cases to enumerate (0 = all)

conditions = ([('and', '--chain-split')] if RUN_AND else []) + ([('hardtraps', '--hard-traps')] if RUN_OR else [])
cells = {name: f'runs/{DATASET}/{name}/{TAG}' for name, _ in conditions}
natural_cell = f'runs/{DATASET}/natural/{TAG}'
print(cells, natural_cell if RUN_NATURAL else '(no natural)')

In [ ]:
# Build + generate + oracle + methods, one condition at a time. Progress streams live.
for name, flag in conditions:
    print(f'==== pipeline: {name} ====', flush=True)
    subprocess.run(
        [sys.executable, 'lineup/scripts/run_pipeline.py',
         '--dataset', DATASET, flag, '--limit', str(LIMIT), '--k', str(K), '--seed', str(SEED),
         '--model', MODEL, '--load-in-4bit', '--out', cells[name]],
        check=True,
    )
    out = Path(cells[name])
    needed = ['scenarios.jsonl', 'generations.jsonl', 'roles.jsonl', 'predictions.jsonl', 'designed.jsonl']
    missing = [f for f in needed if not (out / f).exists()]
    if missing:
        print(f'!!!!!! STOP: {name} missing {missing}')
    else:
        print(f'[ok] {name}: all artifacts written')

In [ ]:
# Does the model behave as the construction designed? Exact minimal sufficient sets vs the plant.
for name, _ in conditions:
    print(f'==== validate_designed: {name} ====', flush=True)
    args = [sys.executable, 'dragnet/scripts/validate_designed.py', '--cell', cells[name],
            '--model', MODEL, '--load-in-4bit', '--max-size', str(VALIDATE_MAX_SIZE)]
    if VALIDATE_LIMIT:
        args += ['--limit', str(VALIDATE_LIMIT)]
    subprocess.run(args, check=True)

In [ ]:
# The five extraction arms: coverage of the designed set vs queries spent.
for name, _ in conditions:
    print(f'==== run_extraction: {name} ====', flush=True)
    args = [sys.executable, 'dragnet/scripts/run_extraction.py', '--cell', cells[name],
            '--model', MODEL, '--load-in-4bit', '--n-samples', str(N_SAMPLES), '--seed', str(SEED)]
    if EXTRACT_LIMIT:
        args += ['--limit', str(EXTRACT_LIMIT)]
    subprocess.run(args, check=True)

In [ ]:
# Organic errors, nothing planted: build the natural cell, then enumerate the exact minimal
# sufficient sets on the live model — the coverage target the guarantee is actually about — and
# log DRAGNET's own interaction/shapley orders so the conformal guarantee (H2) can be scored against
# the coalition-aware ranking, not just the ContextCite baseline.
if RUN_NATURAL:
    print('==== pipeline: natural ====', flush=True)
    subprocess.run(
        [sys.executable, 'lineup/scripts/run_pipeline.py',
         '--dataset', DATASET, '--natural', *(['--real-retriever'] if REAL_RETRIEVER else []),
         '--limit', str(LIMIT), '--k', str(K), '--seed', str(SEED),
         '--model', MODEL, '--load-in-4bit', '--out', natural_cell],
        check=True,
    )
    print('==== run_natural_mscs ====', flush=True)
    args = [sys.executable, 'dragnet/scripts/run_natural_mscs.py', '--cell', natural_cell,
            '--model', MODEL, '--load-in-4bit', '--max-size', str(MSCS_MAX_SIZE)]
    if MSCS_LIMIT:
        args += ['--limit', str(MSCS_LIMIT)]
    subprocess.run(args, check=True)
    if RUN_ORDERS:
        print('==== run_orders (natural) ====', flush=True)
        oargs = [sys.executable, 'dragnet/scripts/run_orders.py', '--cell', natural_cell,
                 '--model', MODEL, '--load-in-4bit', '--n-samples', str(N_SAMPLES), '--seed', str(SEED)]
        if MSCS_LIMIT:
            oargs += ['--limit', str(MSCS_LIMIT)]
        subprocess.run(oargs, check=True)

In [ ]:
# Free check on the AND cells: the designed silent link. If the construction works, the link
# passage should label silent (causal, never salient) and the value passage culprit. Guarded so a
# failure here can never stop the zip in the next cell — every artifact is already on disk.
try:
    from collections import Counter
    from lineup.data.serialization import read_roles, read_scenarios

    if RUN_AND:
        scenarios = {s.qid: s for s in read_scenarios(Path(cells['and']) / 'scenarios.jsonl')}
        link_roles, value_roles = Counter(), Counter()
        for case in read_roles(Path(cells['and']) / 'roles.jsonl'):
            if case.original_correct:
                continue
            provenance = {c.chunk_id: c.provenance for c in scenarios[case.qid].chunks}
            for role in case.chunk_roles:
                if provenance.get(role.chunk_id) == 'link':
                    link_roles[role.role] += 1
                elif provenance.get(role.chunk_id) == 'misleading':
                    value_roles[role.role] += 1
        print('link passage roles :', dict(link_roles))
        print('value passage roles:', dict(value_roles))
except Exception as exc:
    print('link-role sanity check skipped:', repr(exc))

In [ ]:
# Zip everything for download, self-labelled.
import shutil
stem = f'dragnet_{DATASET}_{TAG}_seed{SEED}{SUFFIX}'
shutil.make_archive(stem, 'zip', 'runs')
print('download:', f'/kaggle/working/{stem}.zip')